# DermaFace AI — ResNet50 (transfer learning)

**Owner:** Iva

4-class image classifier for facial skin conditions: **acne, rosacea, redness, clear**.
PyTorch, using the team's real data pipeline from
[neuroarcane/dermaface-ai](https://github.com/neuroarcane/dermaface-ai).

**Run this with the repo's venv as the Jupyter kernel** (`~/dermaface-ai/.venv`) — it imports
the installed `dermaface` package directly, so config/paths/classes all come from
`src/dermaface/config.py`, not a local copy.

Uses `dermaface.models.build_model()` directly — this *is* the team's locked backbone (`cfg.backbone == "resnet50"`), not a separate implementation.

Keep this notebook's data/model plumbing identical to the other two model notebooks (Basic CNN /
ResNet / VGG16) so results are directly comparable.

**Adds over the base pipeline:** best-checkpoint saving (by val macro-F1) to `models/`, a
Word evaluation report to `docs/eval_reports/`, and a push of the best checkpoint to a shared
private Hugging Face model repo.


## Imports

In [ ]:
from collections import Counter
from datetime import date
from pathlib import Path

import torch
from torch import nn
from docx import Document
from huggingface_hub import HfApi, create_repo
from sklearn.metrics import precision_recall_fscore_support

from dermaface.config import CLASS_NAMES, REPO_ROOT, load_config
from dermaface.data.dataset import build_dataloaders
from dermaface.data.weights import class_weights
from dermaface.models import build_model, get_gradcam_target_layer
from dermaface.training.metrics import classification_metrics, confusion, fairness_by_skin_type


## Data

`build_dataloaders()` (owned by Aparna/Rolando) reads the frozen split manifests
(`data/processed/{train,eval,test,demo}_manifest.csv`) and returns ready-to-use
`DataLoader`s. Train is augmented (random-resized-crop, hflip, small rotation, mild
brightness/contrast — saturation/hue are deliberately pinned to 0 since they'd distort the
erythema signal the model needs for redness/rosacea); eval/test/demo are deterministic.

Class imbalance is handled via a **class-weighted loss** (`class_weights()`), not resampling —
so `balance_train` is left at its default `False`. Don't turn it on alongside the weighted
loss below; that double-corrects the imbalance (team decision, see `dermaface.data.weights`).

In [ ]:
cfg = load_config()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loaders = build_dataloaders(cfg)
train_loader, val_loader, test_loader = loaders["train"], loaders["eval"], loaders["test"]

for name, dl in loaders.items():
    n = len(dl.dataset) if dl is not None else 0
    print(f"{name}: {n} samples")

weights = torch.tensor(class_weights(), dtype=torch.float, device=DEVICE)
print(f"classes: {CLASS_NAMES}")
print(f"class weights (balanced, train split): {weights.tolist()}")


## Model

In [ ]:
ARCH_NAME = "resnet50"

# Uses the team's real model factory (dermaface.models.build_model, owned by Iva) —
# cfg.backbone defaults to "resnet50" with pretrained ImageNet weights, so this is
# the currently-locked backbone choice, not a notebook-local reimplementation.
model = build_model(cfg).to(DEVICE)

for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True  # start frozen except the replaced head

print(model.fc)


## Train

Checkpoints the **best validation macro-F1** to `models/dermaface_best_<ARCH_NAME>.pt` — same
convention `train.py`'s own docstring specifies (`checkpoint best val macro-F1 -> cfg.model_path`),
just with a per-architecture filename so the three notebooks don't clobber each other during the
bake-off. Uses `dermaface.training.metrics.classification_metrics` (Iva's implementation) rather
than a separate accuracy calculation, so numbers here match what `evaluate.py` will report later.

In [ ]:
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=cfg.learning_rate
)

CHECKPOINT_PATH = cfg.model_path.parent / f"dermaface_best_{ARCH_NAME}.pt"
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)


def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss, total = 0.0, 0
    y_true, y_pred = [], []
    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            total += images.size(0)
            y_true.extend(labels.tolist())
            y_pred.extend(outputs.argmax(1).tolist())
    result = classification_metrics(y_true, y_pred)
    result["loss"] = total_loss / total
    return result


best_val_macro_f1 = -1.0
best_epoch = 0
for epoch in range(cfg.epochs):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)
    print(
        f"epoch {epoch + 1}/{cfg.epochs}  "
        f"train_loss={train_metrics['loss']:.4f} train_acc={train_metrics['accuracy']:.4f}  "
        f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f} "
        f"val_macro_f1={val_metrics['macro_f1']:.4f}"
    )
    if val_metrics["macro_f1"] > best_val_macro_f1:
        best_val_macro_f1 = val_metrics["macro_f1"]
        best_epoch = epoch + 1
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "arch": ARCH_NAME,
                "val_macro_f1": best_val_macro_f1,
                "epoch": best_epoch,
                "class_names": CLASS_NAMES,
            },
            CHECKPOINT_PATH,
        )
        print(f"  -> new best (val_macro_f1={best_val_macro_f1:.4f}), saved to {CHECKPOINT_PATH}")

print(f"\nBest val macro-F1: {best_val_macro_f1:.4f} (epoch {best_epoch}) — checkpoint at {CHECKPOINT_PATH}")


## Evaluate on the frozen test set

Reloads the best checkpoint (not just whatever's in memory after the last epoch) and scores it
against `docs/requirements.md`'s targets: P1 (beat majority-class baseline), P2 (macro-F1 ≥ 0.60),
P3 (per-class recall ≥ 0.50 each), Fa1 (macro-F1 gap across Fitzpatrick skin-tone bands ≤ 0.15).
Skin-tone bands (I-II / III-IV / V-VI) come straight from `dermaface.config.skin_tone_band`, same
grouping `model-card.md` uses — per-type is too thin to report on its own (type VI is ~2% of the
data).

In [ ]:
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        y_true.extend(labels.tolist())
        y_pred.extend(outputs.argmax(1).cpu().tolist())

skin_bands = [row["skin_tone_band"] for row in test_loader.dataset.rows]

overall = classification_metrics(y_true, y_pred)
per_class_p, per_class_r, per_class_f1, per_class_support = precision_recall_fscore_support(
    y_true, y_pred, labels=list(range(cfg.num_classes)), zero_division=0
)
cm = confusion(y_true, y_pred)
band_metrics = fairness_by_skin_type(y_true, y_pred, skin_bands)

majority_label = Counter(y_true).most_common(1)[0][0]
baseline_acc = sum(1 for t in y_true if t == majority_label) / len(y_true)

macro_f1_by_band = {b: m["macro_f1"] for b, m in band_metrics.items()}
fa1_gap = max(macro_f1_by_band.values()) - min(macro_f1_by_band.values()) if macro_f1_by_band else float("nan")

print(f"Test accuracy: {overall['accuracy']:.4f}   Macro-F1: {overall['macro_f1']:.4f}")
print(f"P1 majority-baseline accuracy: {baseline_acc:.4f}  -> met: {overall['accuracy'] > baseline_acc}")
print(f"P2 Macro-F1 >= 0.60 -> met: {overall['macro_f1'] >= 0.60}")
print(f"P3 per-class recall: {dict(zip(CLASS_NAMES, [round(r, 4) for r in per_class_r]))}  -> met: {bool((per_class_r >= 0.50).all())}")
print(f"Fa1 macro-F1 gap across bands: {fa1_gap:.4f}  -> met: {fa1_gap <= 0.15}")
print(f"Confusion matrix (rows=true, cols=pred, order={CLASS_NAMES}):\n{cm}")


## Word evaluation report

Mirrors the structure already in `docs/model-card.md` (overall metrics, target-vs-actual,
per-class table, fairness-by-band table, confusion matrix) so it can be dropped straight into
the final report. Written to `docs/eval_reports/`. Error-analysis / failure-case sections still
need filling in by hand per `docs/report-guide.md` — this only auto-fills the numbers.

In [ ]:
REPORT_PATH = REPO_ROOT / "docs" / "eval_reports" / f"{ARCH_NAME}_eval_report.docx"
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

doc = Document()
doc.add_heading("DermaFace AI — Evaluation Report", level=0)
doc.add_paragraph(f"Architecture: {ARCH_NAME}    Owner: Iva    Date: {date.today().isoformat()}")
doc.add_paragraph(
    f"Checkpoint: {CHECKPOINT_PATH.name}  "
    f"(best val macro-F1 = {best_val_macro_f1:.4f}, epoch {best_epoch})"
)
doc.add_paragraph(
    "Task: 4-class condition classification (acne / rosacea / redness / clear). "
    "Severity de-scoped for v1 (see docs/severity-decision.md)."
)

doc.add_heading("Overall metrics (frozen test set)", level=1)
t = doc.add_table(rows=1, cols=2)
t.style = "Light Grid Accent 1"
t.rows[0].cells[0].text, t.rows[0].cells[1].text = "Metric", "Value"
for name, val in [
    ("Accuracy", overall["accuracy"]),
    ("Macro-F1", overall["macro_f1"]),
    ("Macro-Precision", overall["macro_precision"]),
    ("Macro-Recall", overall["macro_recall"]),
]:
    row = t.add_row().cells
    row[0].text, row[1].text = name, f"{val:.4f}"

doc.add_heading("Target vs. Actual (docs/requirements.md)", level=1)
t = doc.add_table(rows=1, cols=4)
t.style = "Light Grid Accent 1"
for i, h in enumerate(["Requirement", "Target", "Actual", "Met?"]):
    t.rows[0].cells[i].text = h
target_rows = [
    (
        "P1 Beat majority baseline", "Required",
        f"{overall['accuracy']:.4f} vs baseline {baseline_acc:.4f}",
        str(overall["accuracy"] > baseline_acc),
    ),
    ("P2 Macro-F1 (4-class)", ">= 0.60", f"{overall['macro_f1']:.4f}", str(overall["macro_f1"] >= 0.60)),
    (
        "P3 Per-class recall", ">= 0.50 each",
        ", ".join(f"{c}={r:.2f}" for c, r in zip(CLASS_NAMES, per_class_r)),
        str(bool((per_class_r >= 0.50).all())),
    ),
    ("Fa1 Macro-F1 gap across skin-tone bands", "<= 0.15", f"{fa1_gap:.4f}", str(fa1_gap <= 0.15)),
]
for r in target_rows:
    row = t.add_row().cells
    for i, v in enumerate(r):
        row[i].text = v

doc.add_heading("Per-class results", level=1)
t = doc.add_table(rows=1, cols=5)
t.style = "Light Grid Accent 1"
for i, h in enumerate(["Class", "Precision", "Recall", "F1", "Support"]):
    t.rows[0].cells[i].text = h
for c, p, r, f1, s in zip(CLASS_NAMES, per_class_p, per_class_r, per_class_f1, per_class_support):
    row = t.add_row().cells
    row[0].text, row[1].text, row[2].text, row[3].text, row[4].text = (
        c, f"{p:.4f}", f"{r:.4f}", f"{f1:.4f}", str(int(s))
    )

doc.add_heading("Fairness by Fitzpatrick skin-tone band", level=1)
t = doc.add_table(rows=1, cols=4)
t.style = "Light Grid Accent 1"
for i, h in enumerate(["Band", "Test support", "Accuracy", "Macro-F1"]):
    t.rows[0].cells[i].text = h
for band in sorted(band_metrics):
    m = band_metrics[band]
    row = t.add_row().cells
    row[0].text, row[1].text, row[2].text, row[3].text = (
        band, str(int(m["sample_count"])), f"{m['accuracy']:.4f}", f"{m['macro_f1']:.4f}"
    )

doc.add_heading("Confusion matrix", level=1)
t = doc.add_table(rows=len(CLASS_NAMES) + 1, cols=len(CLASS_NAMES) + 1)
t.style = "Light Grid Accent 1"
t.rows[0].cells[0].text = "true \\ pred"
for j, c in enumerate(CLASS_NAMES):
    t.rows[0].cells[j + 1].text = c
for i, c in enumerate(CLASS_NAMES):
    t.rows[i + 1].cells[0].text = c
    for j in range(len(CLASS_NAMES)):
        t.rows[i + 1].cells[j + 1].text = str(int(cm[i][j]))

doc.add_heading("Notes", level=1)
doc.add_paragraph(
    "Auto-generated by this notebook from a real training run. Metrics computed via "
    "dermaface.training.metrics (Iva's implementation) so numbers match evaluate.py. "
    "Error-analysis / failure-case sections (docs/report-guide.md) still need filling in "
    "by hand before this goes into the final report."
)

doc.save(REPORT_PATH)
print(f"Wrote {REPORT_PATH}")


## Push best checkpoint to Hugging Face

Shared **private** model repo, one file per architecture so all three notebooks can push to the
same place without clobbering each other. Requires you to already be logged in
(`hf auth login`, or an `HF_TOKEN` env var) — this cell never handles the token directly.

In [ ]:
HF_REPO_ID = "varsh26/dermaface-ai"  # confirmed 2026-07-24 (hf auth whoami -> varsh26)

api = HfApi()
create_repo(HF_REPO_ID, private=True, exist_ok=True, repo_type="model")
api.upload_file(
    path_or_fileobj=str(CHECKPOINT_PATH),
    path_in_repo=CHECKPOINT_PATH.name,
    repo_id=HF_REPO_ID,
    repo_type="model",
)
print(f"Uploaded {CHECKPOINT_PATH.name} -> https://huggingface.co/{HF_REPO_ID}/blob/main/{CHECKPOINT_PATH.name}")
